<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [7]</a>'.</span>

In [1]:
import matplotlib.pyplot as pl
import numpy as np
import pandas as pd
import xarray as xr
import math
import pickle
from pathlib import Path

from fair import FAIR
from fair.interface import fill, initialise
from fair.io import read_properties
from pygments.lexers.textfmts import TodotxtLexer

import statsmodels.api as sm
import pmdarima as pm

# Extension csv process

In [2]:
with open("/glade/work/stevenxu/FAIR_models/arima_list_emission_extension.pkl", "rb") as f:
    arima_list = pickle.load(f)

In [3]:
csv_path = "/glade/u/home/stevenxu/FAIRproject/examples/data/calibrated_constrained_ensemble/extensions_1750-2500.csv"
extension_csv = pd.read_csv(csv_path)

startyear = 2023
endyear = 2500

arima_forecasts = {}
no_emission_species = ['Solar',
 'Volcanic',
 'Aerosol-radiation interactions',
 'Aerosol-cloud interactions',
 'Ozone',
 'Light absorbing particles on snow and ice',
 'Stratospheric water vapour',
 'Land use',
 'Equivalent effective stratospheric chlorine',
 'c-C4F8']

for specie in extension_csv["variable"].unique():
    if specie not in no_emission_species:
        model = arima_list[specie]
        arima_value, arima_CI = model.predict(n_periods=endyear-startyear+1, return_conf_int=True)
        arima_forecasts[specie] = [arima_value, arima_CI[:,0], arima_CI[:,1]]



In [4]:
years_before = [y + 0.5 for y in range(1750, startyear)]    
years_after  = [y + 0.5 for y in range(startyear, endyear + 1)]
arima_scenarios = ['ARIMA_mean', 'ARIMA_low', 'ARIMA_high']
region = "World"


rows = []
for specie in extension_csv["variable"].unique():
    print(specie)
    for scenario_index, scenario in enumerate(arima_scenarios):
        unit = extension_csv.loc[extension_csv["variable"] == specie, "unit"].iloc[0]
        row = [scenario, 'World', specie, unit]
        for year in years_before:
            value = extension_csv.loc[extension_csv["variable"] == specie, str(year)].iloc[0]
            row.append(value)
        for year_index, year in enumerate(years_after):
            if specie in arima_forecasts.keys():
                value = arima_forecasts[specie][scenario_index][year_index]
            else:
                value = extension_csv.loc[extension_csv["variable"] == specie, str(year)].iloc[0]
            row.append(value)
        rows.append(row)

BC
C2F6
C3F8
C4F10
C5F12
C6F14
C7F16
C8F18
CCl4
CF4
CFC-11
CFC-113
CFC-114
CFC-115
CFC-12
CH2Cl2
CH3Br
CH3CCl3
CH3Cl
CH4
CHCl3
CO
CO2 AFOLU
CO2 FFI
HCFC-141b
HCFC-142b
HCFC-22
HFC-125
HFC-134a
HFC-143a
HFC-152a
HFC-227ea
HFC-23
HFC-236fa
HFC-245fa
HFC-32
HFC-365mfc
HFC-4310mee
Halon-1211
Halon-1301
Halon-2402
N2O
NF3


NH3
NOx
OC
SF6
SO2F2
Sulfur
VOC
c-C4F8


In [5]:
rowdf = pd.DataFrame(rows)
rowdf.columns = extension_csv.columns
extension_csv = pd.concat([extension_csv, rowdf], ignore_index=True)
extension_csv

,scenario,region,variable,unit,1750.5,1751.5,1752.5,1753.5,1754.5,1755.5,...,2491.5,2492.5,2493.5,2494.5,2495.5,2496.5,2497.5,2498.5,2499.5,2500.5
0,high-overshoot,World,BC,Mt BC/yr,2.096766,2.071972,2.067178,2.070382,2.098586,2.097789,...,1.081420,1.081420,1.081420,1.081420,1.081420,1.081420,1.081420,1.081420,1.081420,1.081420
1,high-overshoot,World,C2F6,kt C2F6/yr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,high-overshoot,World,C3F8,kt C3F8/yr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,high-overshoot,World,C4F10,kt C4F10/yr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,high-overshoot,World,C5F12,kt C5F12/yr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,ARIMA_low,World,VOC,Mt VOC/yr,60.622840,59.691733,59.366595,59.634416,60.880206,60.436960,...,-118.280876,-118.956883,-121.998257,-115.626141,-109.307814,-122.227588,-123.023651,-117.482716,-122.521629,-131.239395
506,ARIMA_high,World,VOC,Mt VOC/yr,60.622840,59.691733,59.366595,59.634416,60.880206,60.436960,...,1196.891624,1200.033957,1200.799901,1210.968437,1221.072302,1211.927324,1214.895408,1224.189930,1222.894130,1217.909093
507,ARIMA_mean,World,c-C4F8,kt cC4F8/yr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
508,ARIMA_low,World,c-C4F8,kt cC4F8/yr,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [6]:
for specie in extension_csv["variable"].unique():
    if len(extension_csv[extension_csv['variable'] == specie]) != 10:
        raise IndexError('Scenarios not added in')

extension_csv.to_csv("/glade/u/home/stevenxu/FAIRproject/examples/data/calibrated_constrained_ensemble/extensions_1750-2500_with_arima.csv", index=False)

# Model fitting

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [7]:
f = FAIR(ch4_method="Thornhill2021")
f.define_time(1750, 2300, 1)  # start, end, step

scenarios = [
    "high-extension",
    "high-overshoot",
    "medium-overshoot",
    "medium-extension",
    "low",
    "verylow",
    "verylow-overshoot",
    'ARIMA_mean', 'ARIMA_low', 'ARIMA_high'
]

#scenarios = ['ssp119', 'ssp126', 'ssp245', 'ssp370', 'ssp434', 'ssp460', 'ssp534-over', 'ssp585']
f.define_scenarios(scenarios)
fair_params_1_4_1_file = '/glade/u/home/stevenxu/FAIRproject/examples/data/calibrated_constrained_ensemble/calibrated_constrained_parameters_calibration1.4.1.csv'
df_configs = pd.read_csv(fair_params_1_4_1_file, index_col=0)
configs = df_configs.index  # this is used as a label for the "config" axis
f.define_configs(configs)
fair_species_configs_1_4_1_file = '/glade/u/home/stevenxu/FAIRproject/examples/data/calibrated_constrained_ensemble/species_configs_properties_calibration1.4.1.csv'
species, properties = read_properties(filename=fair_species_configs_1_4_1_file)
f.define_species(species, properties)
f.allocate()
f.fill_from_csv(
    emissions_file="/glade/u/home/stevenxu/FAIRproject/examples/data/calibrated_constrained_ensemble/extensions_1750-2500_with_arima.csv",
    forcing_file='/glade/u/home/stevenxu/FAIRproject/examples/data/calibrated_constrained_ensemble/volcanic_solar.csv',
)

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
fill(
    f.forcing,
    f.forcing.sel(specie="Volcanic") * df_configs["forcing_scale[Volcanic]"].values.squeeze(),
    specie="Volcanic",
)
fill(
    f.forcing,
    f.forcing.sel(specie="Solar") * df_configs["forcing_scale[Solar]"].values.squeeze(),
    specie="Solar",
)
f.fill_species_configs(fair_species_configs_1_4_1_file)
f.override_defaults(fair_params_1_4_1_file)
initialise(f.concentration, f.species_configs["baseline_concentration"])
initialise(f.forcing, 0)
initialise(f.temperature, 0)
initialise(f.cumulative_emissions, 0)
initialise(f.airborne_emissions, 0)
initialise(f.ocean_heat_content_change, 0)

# Run

In [ ]:
f.run()

In [ ]:
fancy_titles = {
    'high-extension': 'High extension',
    'high-overshoot': 'High overshoot',
    'medium-extension': 'Medium extension',
    'medium-overshoot': 'Medium overshoot',
    'low': 'Low',
    'verylow': 'Very low',
    'verylow-overshoot': 'Very low overshoot',
    'ARIMA_mean': 'ARIMA mean',
    'ARIMA_low': 'ARIMA low',
    'ARIMA_high': 'ARIMA high',
}

colors = {
    'high-extension': '#800000',   # dark red
    'high-overshoot': '#ff0000',   # bright red
    'medium-extension': '#c87820', # orange-brown
    'medium-overshoot': '#d3a640', # gold
    'low': '#098740',              # green
    'verylow': '#0080d0',          # blue
    'verylow-overshoot': '#100060',# navy
    'ARIMA_mean': '#6a0dad',       # royal purple (distinct mean)
    'ARIMA_low': '#9370db',        # lighter purple (low)
    'ARIMA_high': '#4b0082',       # indigo (high)
}


In [ ]:
weights_51yr = np.ones(52)
weights_51yr[0] = 0.5
weights_51yr[-1] = 0.5

In [ ]:
fig, ax = pl.subplots(2, 4, figsize=(12, 6))

for i, scenario in enumerate(scenarios):
    for pp in ((0, 100), (5, 95), (16, 84)):
        ax[i // 4, i % 4].fill_between(
            f.timebounds,
            np.percentile(
                f.temperature.loc[dict(scenario=scenario, layer=0)]
                - np.average(
                    f.temperature.loc[
                        dict(scenario=scenario, timebounds=np.arange(1850, 1902), layer=0)
                    ],
                    weights=weights_51yr,
                    axis=0
                ),
                pp[0],
                axis=1,
            ),
            np.percentile(
                f.temperature.loc[dict(scenario=scenario, layer=0)]
                - np.average(
                    f.temperature.loc[
                        dict(scenario=scenario, timebounds=np.arange(1850, 1902), layer=0)
                    ],
                    weights=weights_51yr,
                    axis=0
                ),
                pp[1],
                axis=1,
            ),
            color=colors[scenarios[i]],
            alpha=0.2,
            lw=0
        )

    ax[i // 4, i % 4].plot(
        f.timebounds,
        np.median(
            f.temperature.loc[dict(scenario=scenario, layer=0)]
            - np.average(
                f.temperature.loc[
                    dict(scenario=scenario, timebounds=np.arange(1850, 1902), layer=0)
                ],
                weights=weights_51yr,
                axis=0
            ),
            axis=1,
        ),
        color=colors[scenarios[i]],
    )
    ax[i // 4, i % 4].set_xlim(1850, 2300)
    ax[i // 4, i % 4].set_ylim(-1, 10)
    ax[i // 4, i % 4].axhline(0, color="k", ls=":", lw=0.5)
    ax[i // 4, i % 4].set_title(fancy_titles[scenarios[i]])

pl.suptitle("Temperature anomalies")
fig.tight_layout()